In [1]:
import sys
from pathlib import Path
# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

In [2]:
from synth_extract.agents.llm import LLMBackend  # noqa: E402


from pydantic import BaseModel, ConfigDict, Field, field_validator
from synth_extract.agents.classification.schemas import CompletionMetadata

In [3]:
import pandas as pd
import numpy as np

In [4]:
import asyncio

In [5]:
class PropertyExtractionResult(BaseModel):
    """Verbatim measured-property names extracted from a passage."""

    model_config = ConfigDict(extra="forbid", strict=True)

    properties: list[str] = Field(
        description=(
            "Distinct property-name spans for which a corresponding property "
            "value is explicitly reported in the passage. Each string must be "
            "copied exactly from the passage. Do not include values, units, "
            "measurement methods, conditions, materials, or inferred properties."
        )
    )

    metadata: CompletionMetadata

    @field_validator("properties")
    @classmethod
    def validate_properties(cls, properties: list[str]) -> list[str]:
        """Require non-empty, distinct spans after model generation."""
        if any(not property_name.strip() for property_name in properties):
            raise ValueError("Property spans must be non-empty strings.")
        if len(properties) != len(set(properties)):
            raise ValueError("Property spans must be distinct.")
        return properties

In [13]:
import json
from pathlib import Path
from typing import Any


class PropertyDiscoveryLLM:
    """Extract measured-property spans from a paper using an LLMBackend."""

    def __init__(
        self,
        backend: LLMBackend,
        system_prompt_path: str | Path | None = None,
        user_prompt_path: str | Path | None = None,
    ) -> None:
        self.backend = backend
        if system_prompt_path is None or user_prompt_path is None:
            prompt_dir = self._find_prompt_dir()
        self.system_prompt_path = (
            Path(system_prompt_path)
            if system_prompt_path is not None
            else prompt_dir / "property_discovery.md"
        )
        self.user_prompt_path = (
            Path(user_prompt_path)
            if user_prompt_path is not None
            else prompt_dir / "user_prompt.md"
        )
        self.reload_prompts()

    @staticmethod
    def _find_prompt_dir() -> Path:
        """Locate prompts when Jupyter starts here or at the project root."""
        candidates = (Path.cwd(), Path.cwd() / "discovery")
        for candidate in candidates:
            if (candidate / "property_discovery.md").is_file() and (
                candidate / "user_prompt.md"
            ).is_file():
                return candidate
        raise FileNotFoundError(
            "Could not find property_discovery.md and user_prompt.md. "
            "Pass their paths explicitly."
        )

    def reload_prompts(self) -> None:
        """Reload both UTF-8 prompt files from disk."""
        self._system_prompt = self.system_prompt_path.read_text(
            encoding="utf-8"
        ).strip()
        self._user_prompt = self.user_prompt_path.read_text(
            encoding="utf-8"
        ).strip()

    @staticmethod
    def response_format() -> dict[str, Any]:
        """Require the bare string array specified by the system prompt.

        Uniqueness is validated locally because the endpoint's grammar
        compiler does not implement JSON Schema's ``uniqueItems`` keyword.
        """
        return {
            "type": "json_schema",
            "json_schema": {
                "name": "measured_property_spans",
                "strict": True,
                "schema": {
                    "type": "array",
                    "items": {"type": "string"},
                },
            },
        }

    def build_messages(self, fulltext: str) -> list[dict[str, str]]:
        """Build the messages for one paper without calling the model."""
        if not isinstance(fulltext, str) or not fulltext.strip():
            raise ValueError("fulltext must be a non-empty string")
        return [
            {"role": "system", "content": self._system_prompt},
            {
                "role": "user",
                "content": self._user_prompt.format(
                    fulltext=fulltext.strip()
                ),
            },
        ]

    def render_request(self, fulltext: str) -> str:
        """Render the exact request body without sending it."""
        return self.backend.render_request(
            messages=self.build_messages(fulltext),
            response_format=self.response_format(),
        )

    @staticmethod
    def _metadata(completion: Any) -> CompletionMetadata:
        choice = completion.choices[0]
        message = choice.message
        usage = getattr(completion, "usage", None)
        token_usage = None
        if usage is not None:
            token_usage = {
                "prompt_tokens": getattr(usage, "prompt_tokens", None),
                "completion_tokens": getattr(
                    usage, "completion_tokens", None
                ),
                "total_tokens": getattr(usage, "total_tokens", None),
            }
        return CompletionMetadata(
            model=getattr(completion, "model", None),
            created=getattr(completion, "created", None),
            finish_reason=getattr(choice, "finish_reason", None),
            stop_reason=getattr(choice, "stop_reason", None),
            reasoning=getattr(message, "reasoning", None),
            usage=token_usage,
        )

    @classmethod
    def _parse_completion(
        cls, completion: Any, fulltext: str
    ) -> PropertyExtractionResult:
        if not completion.choices:
            raise ValueError("The provider returned no completion choices.")

        choice = completion.choices[0]
        if choice.finish_reason == "length":
            raise ValueError("The property response reached the token limit.")

        refusal = getattr(choice.message, "refusal", None)
        if refusal:
            raise ValueError(f"The provider refused the request: {refusal}")

        content = choice.message.content
        if not content:
            raise ValueError("The provider returned an empty response.")

        properties = json.loads(content)
        if not isinstance(properties, list):
            raise ValueError("The response must be a JSON array.")
        if not all(isinstance(item, str) for item in properties):
            raise ValueError("Every property span must be a string.")
        missing = [
            item
            for item in properties
            if item not in fulltext
        ]
        if missing:
            raise ValueError(
                f"Property spans not found verbatim in the paper: {missing}"
            )

        return PropertyExtractionResult(
            properties=properties,
            metadata=cls._metadata(completion),
        )

    def extract_raw(self, fulltext: str) -> Any:
        """Synchronously return the backend's raw completion."""
        return self.backend.create_completion(
            messages=self.build_messages(fulltext),
            response_format=self.response_format(),
        )

    def extract(self, fulltext: str) -> PropertyExtractionResult:
        """Synchronously extract and validate properties from a paper."""
        return self._parse_completion(self.extract_raw(fulltext), fulltext)

    async def aextract_raw(self, fulltext: str) -> Any:
        """Asynchronously return the backend's raw completion."""
        return await self.backend.acreate_completion(
            messages=self.build_messages(fulltext),
            response_format=self.response_format(),
        )

    async def aextract(self, fulltext: str) -> PropertyExtractionResult:
        """Asynchronously extract and validate properties from a paper."""
        completion = await self.aextract_raw(fulltext)
        return self._parse_completion(completion, fulltext)

    def health_check(self) -> bool:
        """Return ``True`` when the endpoint responds to a model-list request."""
        self.backend.list_models()
        return True

In [9]:
host="127.0.0.1"
port="8000"
base_url=f"http://{host}:{port}/v1"

model = "qwen3.6-27b"
api_key = "none"

max_tokens=8192
extra_body = {"chat_template_kwargs":{"enable_thinking":False}}

backend = LLMBackend(
    model=model,
    base_url=base_url,
    api_key=api_key,
    temperature=0.0,
    timeout=300,
    max_tokens=max_tokens,
    extra_body=extra_body,
)

In [10]:
base_path = Path("/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/discovery")
system_prompt_path = base_path / "property_discovery.md"
user_prompt_path = base_path / "user_prompt.md"

In [14]:
discoveryLM = PropertyDiscoveryLLM(backend=backend,
                                   system_prompt_path=system_prompt_path,
                                   user_prompt_path=user_prompt_path)

In [15]:
discoveryLM.health_check()

True

In [16]:
full_text_path = Path("/nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/data/fulltext")

In [17]:
# Pick a paper
uid = "ID000403571"
source = "elsevier"

# Construct full-text path
fulltext_path = (
    Path(full_text_path)
    / source
    / uid
    / f"{uid}.md"
)

print("Source:", source)
print("Full text:", fulltext_path)

if not fulltext_path.exists():
    raise FileNotFoundError(fulltext_path)

# Load Markdown
full_text = fulltext_path.read_text(encoding="utf-8")

print(f"Characters: {len(full_text):,}")

# Classify
result = discoveryLM.extract(full_text)

result

Source: elsevier
Full text: /nobackup/proj/disk/naiss2024-5-630/personal/george/synth_extract/data/fulltext/elsevier/ID000403571/ID000403571.md
Characters: 55,596


LLMBackendError: HTTP 400: Error code: 400 - {'error': {'message': 'Grammar error: Unimplemented keys: ["uniqueItems"]', 'type': 'BadRequestError', 'param': None, 'code': 400}}